# 14 — Noise injection on the H2 emulators

Can the trade-off be shown on Quantinuum's own emulators, with their full gate, readout and
leakage noise? The memory program injects the paper's data error itself (one seeded RNG per
shot), so Δt is set by the program, not by the machine's unknown round time. Two steps: a small
d = 3 check that injection works end to end, then a d = 5 Δt sweep.

**Runs.** aqora as organisation `q_sq`; H2-1E (d = 3, 100 shots) and H2-2E (d = 5, 1000 shots per
point). Every program syntax-checked first. Results: `results/hardware/h2_1e_memory_d3.json`,
`h2_2e_dt_sweep_d5_circuit_decoder.json`, `h2_2e_dt_sweep_d5_model_curves.json`.

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from qec_timing import plotting as P
P.style()
R = Path("..") / "results"

In [ ]:
from scipy.stats import beta
cp = lambda k, n: (beta.ppf(.025, k, n - k + 1) if k else 0.0, beta.ppf(.975, k + 1, n - k))
mem = json.loads((R / "hardware" / "h2_1e_memory_d3.json").read_text())
for name, r in mem["runs"].items():
    print(f"{name:<15} injected p {r['injected_p']:<5} raw logical flips {r['raw_logical_flips']}/{r['shots']}, "
          f"decoded failures {r['decoded_failures']}/{r['shots']}" + (f", Stim {r['stim_pL']:.3f}" if "stim_pL" in r else ""))

## Step 1: injection works end to end (d = 3, 2 rounds, H2-1E)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [2.2, 1]})
labels = [f"r{r} c{c}" for r in range(2) for c in range(4)] + [f"final c{c}" for c in range(4)]
x = np.arange(12); w = 0.38
for i, (name, lab, col) in enumerate((("d3_X_r2_p0", "emulator noise only", P.SERIES[0]),
                                      ("d3_X_r2_p0.05", "+ injected Z, p = 0.05", P.SERIES[1]))):
    r = mem["runs"][name]; dr = np.array(r["detector_rates"]); n = r["shots"]
    ci = np.array([cp(int(round(v * n)), n) for v in dr]).T
    axes[0].bar(x + (i - .5) * w, dr, w, color=col, alpha=.85, label=lab)
    axes[0].errorbar(x + (i - .5) * w, dr, yerr=[dr - ci[0], ci[1] - dr], fmt="none", ecolor=P.INK_2, lw=.8)
    if "stim_detector_rates" in r:
        axes[0].plot(x + (i - .5) * w, r["stim_detector_rates"], "_", ms=14, mew=2, color=P.INK, label="Stim, injection only")
    for j, (k, tag) in enumerate(((r["raw_logical_flips"], "undecoded"), (r["decoded_failures"], "decoded"))):
        lo, hi = cp(k, n); xx = i * 2.4 + j
        axes[1].errorbar(xx, k / n, yerr=[[k / n - lo], [hi - k / n]], fmt="s" if j == 0 else "o", color=col, capsize=3)
    if "stim_pL" in r:
        axes[1].plot([i * 2.4 + .7, i * 2.4 + 1.3], [r["stim_pL"]] * 2, color=P.INK, lw=2)
axes[0].set_xticks(x, labels, rotation=45, ha="right", fontsize=8); axes[0].set_ylabel("detection probability")
axes[0].legend(fontsize=8); axes[0].set_title("Detectors (100 shots, 95% CI)", loc="left")
axes[1].set_xticks([0, 1, 2.4, 3.4], ["p0\nraw", "p0\ndecoded", "p0.05\nraw", "p0.05\ndecoded"], fontsize=8)
axes[1].set_ylabel("logical X failure"); axes[1].set_title("Logical error (black: Stim)", loc="left")
fig.tight_layout()
P.save(fig, "14_emulator_noise_injection_1")

**Reading.**
- *Measured:* with injection, every detector and the decoded logical error (6/100 against Stim's
  7.2%) agree with Stim within the statistics: noise written into the program behaves the same on
  the emulator as in simulation.
- *Measured:* with no injection, the raw data show no logical flip, but decoding on the paper's
  graph produces 2/100: the emulator's own errors are not on that graph. Step 2 decodes with a
  circuit model of the program instead.

## Step 2: the Δt sweep at d = 5 (H2-2E)

d = 5, X basis, fixed T = 10, R = T/Δt rounds; before each round every data qubit gets Z with the
paper's p_data(Δt) (p = 0.015, λ = 1); record flips p_read = 0.015 are added to the returned bits
afterwards (a classical flip, identical to doing it in the program). Two references: the paper's
model alone, and a Stim model of the program gate for gate (one reused ancilla, serial checks)
with H2-2E's published gate errors (`hardware/program_model.py`), whose detector error model also
decodes the emulator's shots.

In [ ]:
cur = json.loads((R / "hardware" / "h2_2e_dt_sweep_d5_model_curves.json").read_text())
dec = json.loads((R / "hardware" / "h2_2e_dt_sweep_d5_circuit_decoder.json").read_text())["rows"]
pts = [r for r in dec if "h22e_program_dem_H22E" in r]
print(f"{'dt':>5}{'emulator':>10}{'model + H2-2E gates':>21}{'paper model':>13}")
for r in dec:
    e = r.get("h22e_program_dem_H22E")
    print(f"{r['dt']:>5.2g}{(f'{e:.3f}' if e is not None else 'missing'):>10}{r['stim_program_with_H22E_gates']:>21.3f}{r['stim_program_injected_only']:>13.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
dts = [10 / r for r in cur["R"]]
ax.plot(dts, cur["curves"]["injected only"], "-", color=P.MUTED, lw=2, label="paper model (injected noise only)")
ax.plot(dts, cur["curves"]["injected + H2-2E gate/readout noise"], "--", color=P.SERIES[0], lw=2,
        label="circuit model of the program + H2-2E gate noise")
x = np.array([r["dt"] for r in pts]); y = np.array([r["h22e_program_dem_H22E"] for r in pts])
ci = np.array([cp(int(round(v * 1000)), 1000) for v in y]).T
ax.errorbar(x, y, yerr=[y - ci[0], ci[1] - y], fmt="o", color=P.SERIES[1], ms=7, capsize=4,
            label="H2-2E emulator (1000 shots, 95% CI)")
ax.set_xscale("log"); ax.set_xticks([0.25, 0.5, 1, 2, 5], ["0.25", "0.5", "1", "2", "5"])
ax.xaxis.set_minor_formatter(plt.NullFormatter())
ax.set_xlabel(r"interval $\Delta t$ ($T$ = 10; $R = T/\Delta t$ rounds)"); ax.set_ylabel(r"$p_L$ at $T$ = 10")
ax.legend(fontsize=8, loc="upper center")
ax.set_title("d = 5: the emulator follows the circuit model with its gate noise", loc="left")
fig.tight_layout()
P.save(fig, "14_emulator_noise_injection_2")

**Reading.**
- *Measured:* the three returned points agree with the circuit model including H2-2E's gate noise
  within about 1σ (+1.2σ, 0.0σ, −1.4σ); against the paper model alone the Δt = 1 point is 4σ high.
- *Interp.:* the emulator adds its gate and readout error once per round, so frequent extraction
  costs more than in the paper's model: the U-curve is higher and flatter, minimum ≈ 0.16 near
  Δt ≈ 1, instead of ≈ 0.13. The trade-off survives device-grade noise.
- *Missing:* Δt = 0.5 (R = 20) ran to completion but aqora never returned its result; Δt = 0.14
  (R = 70) was refused as over the per-job cost limit, so the left arm is model only.
- *Not in the model:* idle dephasing in real time and leakage.